# 01 · Fuentes y descarga de datos

En este notebook se descargan las tres fuentes oficiales de NYC Open Data que usa el proyecto:

| Archivo | Contenido |
|---|---|
| `tonnage.csv` | Toneladas mensuales recolectadas por distrito (DSNY) |
| `districts.geojson` | Límites de los distritos comunitarios (DCP) |
| `population.csv` | Población por distrito, censos 1970-2010 (DCP) |

Si los archivos ya existen en `data/raw/`, no se vuelven a descargar. Así los datos originales no se modifican.

In [1]:
from pathlib import Path
import pandas as pd

# Carpeta del proyecto (funciona si el notebook se abre desde la raíz o desde notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'data' / 'processed'
FIG = ROOT / 'docs' / 'figures'

## Descarga

In [2]:
import requests

BASE = 'https://data.cityofnewyork.us'
FUENTES = {
    'tonnage.csv':       BASE + '/api/views/ebb7-mvp5/rows.csv?accessType=DOWNLOAD',
    'districts.geojson': BASE + '/resource/5crt-au7u.geojson?$limit=5000',
    'population.csv':    BASE + '/api/views/xi7c-iiu2/rows.csv?accessType=DOWNLOAD',
    # Metadatos: autor, fecha de actualización y descripción de columnas
    'tonnage_metadata.json':    BASE + '/api/views/ebb7-mvp5.json',
    'districts_metadata.json':  BASE + '/api/views/5crt-au7u.json',
    'population_metadata.json': BASE + '/api/views/xi7c-iiu2.json',
}

RAW.mkdir(parents=True, exist_ok=True)
for nombre, url in FUENTES.items():
    ruta = RAW / nombre
    if not ruta.exists():
        respuesta = requests.get(url, timeout=180)
        respuesta.raise_for_status()
        ruta.write_bytes(respuesta.content)
    print(f'{nombre:28} {ruta.stat().st_size / 1e6:6.2f} MB')

tonnage.csv                    1.22 MB
districts.geojson              3.79 MB
population.csv                 0.00 MB
tonnage_metadata.json          0.03 MB
districts_metadata.json        0.01 MB
population_metadata.json       0.02 MB


## Origen y condiciones de uso

Los metadatos indican qué institución publica cada conjunto y cuándo se actualizó por última vez. Los datos son públicos y se reutilizan bajo los Términos de Uso de NYC Open Data, citando la fuente.

In [3]:
import json
from datetime import datetime

filas = []
for fuente in ['tonnage', 'districts', 'population']:
    meta = json.loads((RAW / f'{fuente}_metadata.json').read_text(encoding='utf-8'))
    filas.append({'conjunto': meta['name'],
                  'publica': meta.get('attribution'),
                  'ultima_actualizacion': datetime.fromtimestamp(meta['rowsUpdatedAt']).date()})
pd.DataFrame(filas)

,conjunto,publica,ultima_actualizacion
0,DSNY Monthly Tonnage Data,Department of Sanitation (DSNY),2026-11-02
1,Community Districts,Department of City Planning (DCP),2026-11-02
2,New York City Population By Community Districts,Department of City Planning (DCP),2024-11-12


## Primera mirada a los datos en bruto

In [4]:
tonelaje = pd.read_csv(RAW / 'tonnage.csv')
poblacion = pd.read_csv(RAW / 'population.csv')
print('Tonelaje:', tonelaje.shape, '| Población:', poblacion.shape)
tonelaje.head()

Tonelaje: (25355, 12) | Población: (59, 8)


,MONTH,BOROUGH,COMMUNITYDISTRICT,REFUSETONSCOLLECTED,PAPERTONSCOLLECTED,MGPTONSCOLLECTED,RESORGANICSTONS,SCHOOLORGANICTONS,LEAVESORGANICTONS,XMASTREETONS,OTHERORGANICSTONS,BOROUGH_ID
0,2026 / 10,Bronx,1,3510.3,242.4,126.2,23.9,85.6,NaN,NaN,NaN,2
1,2026 / 10,Bronx,2,8104.7,293.9,254.6,21.1,86.3,NaN,NaN,NaN,2
2,2026 / 10,Bronx,3,2283.0,147.3,132.2,21.7,120.7,NaN,NaN,NaN,2
3,2026 / 10,Bronx,4,4354.0,248.5,275.3,24.3,NaN,NaN,NaN,NaN,2
4,2026 / 10,Bronx,5,3699.8,252.8,305.1,21.0,79.8,NaN,NaN,NaN,2


In [5]:
poblacion.head()

,Borough,CD Number,CD Name,1970 Population,1980 Population,1990 Population,2000 Population,2010 Population
0,Bronx,1,"Melrose, Mott Haven, Port Morris",138557,78441,77214,82159,91497
1,Bronx,2,"Hunts Point, Longwood",99493,34399,39443,46824,52246
2,Bronx,3,"Morrisania, Crotona Park East",150636,53635,57162,68574,79762
3,Bronx,4,"Highbridge, Concourse Village",144207,114312,119962,139563,146441
4,Bronx,5,"University Hts., Fordham, Mt. Hope",121807,107995,118435,128313,128200


In [6]:
import geopandas as gpd
distritos = gpd.read_file(RAW / 'districts.geojson')
print('Polígonos:', len(distritos))
distritos.head(3)

Polígonos: 71


,boro_cd,shape_leng,shape_area,geometry
0,226,32824.8378966,50565041.6084,"MULTIPOLYGON (((-73.8679 40.90294, -73.86796 4..."
1,313,65789.3560939,88264690.3446,"MULTIPOLYGON (((-73.98372 40.59582, -73.98305 ..."
2,105,35288.334228,43790040.2362,"MULTIPOLYGON (((-73.97301 40.76428, -73.97141 ..."


Hay 71 polígonos, pero solo 59 son distritos comunitarios con datos de recolección. Los 12 restantes son parques, aeropuertos y otras áreas sin población residente (por ejemplo 164 Central Park, 226-228, 355-356, 480-484 y 595).

## Diccionario oficial del DSNY
El diccionario explica qué mide cada columna. Se revisan solo las tres fracciones que analiza el proyecto.

In [7]:
import warnings
warnings.filterwarnings('ignore')  # avisos de formato del Excel, no afectan la lectura

dicc = pd.read_excel(next(RAW.glob('*Dictionary*.xlsx')), sheet_name='Column Information', header=None)
columnas = ['REFUSETONSCOLLECTED', 'PAPERTONSCOLLECTED', 'MGPTONSCOLLECTED']
for _, fila in dicc[dicc[0].isin(columnas)].iterrows():
    print(fila[0], '->', fila[1], '\n')

REFUSETONSCOLLECTED -> Tons of trash or refuse collected per month from NYC residences and institutions serviced by the NYC Department of Sanitation. 

PAPERTONSCOLLECTED -> Tons of source-separated recyclable paper collected per month from NYC residences and institutions serviced by the NYC Department of Sanitation. 

MGPTONSCOLLECTED -> Tons of source-separated recyclable metal, glass, plastic, and beverage cartons (MGP) collected per month from NYC residences and institutions serviced by the NYC Department of Sanitation. 



**Conclusión:** las tres fuentes se descargaron completas. El tonelaje se une con población y mapas mediante el código de distrito (`boro_cd`), que se construye en el notebook 02.